In [1]:
# Cell 1: Uninstall conflicting and existing packages for a clean slate
!pip uninstall -y jax ml-dtypes dopamine-rl gymnasium tf-keras tensorflow-decision-forests tensorflow-text tensorstore tensorflow keras keras-nlp

Found existing installation: jax 0.5.2
Uninstalling jax-0.5.2:
  Successfully uninstalled jax-0.5.2
Found existing installation: ml-dtypes 0.4.1
Uninstalling ml-dtypes-0.4.1:
  Successfully uninstalled ml-dtypes-0.4.1
Found existing installation: dopamine_rl 4.1.2
Uninstalling dopamine_rl-4.1.2:
  Successfully uninstalled dopamine_rl-4.1.2
Found existing installation: gymnasium 0.29.0
Uninstalling gymnasium-0.29.0:
  Successfully uninstalled gymnasium-0.29.0
Found existing installation: tf_keras 2.18.0
Uninstalling tf_keras-2.18.0:
  Successfully uninstalled tf_keras-2.18.0
Found existing installation: tensorflow_decision_forests 1.11.0
Uninstalling tensorflow_decision_forests-1.11.0:
  Successfully uninstalled tensorflow_decision_forests-1.11.0
Found existing installation: tensorflow-text 2.18.1
Uninstalling tensorflow-text-2.18.1:
  Successfully uninstalled tensorflow-text-2.18.1
Found existing installation: tensorstore 0.1.74
Uninstalling tensorstore-0.1.74:
  Successfully uninstall

In [2]:


# Install known compatible versions
!pip install -q tensorflow==2.15.0
!pip install -q keras==2.15.0
!pip install -q keras-nlp==0.8.2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.3/475.3 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.3/465.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.9/644.9 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 92.4 MB/s eta 0:00:00


In [3]:



# Cell 2: Import all necessary libraries
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
import keras_nlp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

print("✅ Libraries imported successfully.")

# Cell 3: Load the data
try:
    train_df = pd.read_csv("/kaggle/input/map-charting-student-math-misunderstandings/train.csv")
    test_df = pd.read_csv("/kaggle/input/map-charting-student-math-misunderstandings/test.csv")
    print("✅ Data loaded successfully.")
except FileNotFoundError:
    print("🛑 Data files not found. Make sure the competition data is added to your notebook.")

2025-09-09 03:40:21.442337: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757389221.462993      19 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757389221.469729      19 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1757389221.487625      19 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1757389221.487643      19 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1757389221.487646      19 computation_placer.cc:177] computation placer alr

✅ Libraries imported successfully.
✅ Data loaded successfully.


In [4]:
# Cell 4: Prepare Text and Labels

# 1. Combine text fields into a single input string
train_df['input_text'] = train_df.apply(lambda row: f"Question: {row.QuestionText} Answer: {row.MC_Answer} Explanation: {row.StudentExplanation}", axis=1)
test_df['input_text'] = test_df.apply(lambda row: f"Question: {row.QuestionText} Answer: {row.MC_Answer} Explanation: {row.StudentExplanation}", axis=1)

# 2. Prepare labels for multi-label classification
# Ensure columns are strings to prevent errors
train_df['Category'] = train_df['Category'].astype(str)
train_df['Misconception'] = train_df['Misconception'].astype(str)
train_df['full_label'] = train_df['Category'] + ':' + train_df['Misconception']

# Use MultiLabelBinarizer to create multi-hot encoded vectors
mlb = MultiLabelBinarizer()
y_train = mlb.fit_transform(train_df['full_label'].apply(lambda x: [x]))

print("✅ Text and labels prepared.")

✅ Text and labels prepared.


In [5]:
# Cell 5: Create a Train/Validation Split
X_train, X_val, y_train, y_val = train_test_split(
    train_df['input_text'], 
    y_train, 
    test_size=0.2, 
    random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")

Training samples: 29356
Validation samples: 7340


In [6]:
MODEL_PATH = "/kaggle/input/deberta_v3/keras/deberta_v3_base_en/3"

# Calculate the number of unique labels
num_labels = len(mlb.classes_)

# Load the entire classifier in one go. 
# It automatically includes the preprocessor and a classification head.
classifier = keras_nlp.models.DebertaV3Classifier.from_preset(
    "deberta_v3_base_en",
    num_classes=num_labels,
)

# Compile the model. By using "binary_crossentropy" as the loss, Keras
# will automatically use a 'sigmoid' activation, which is correct for multi-label tasks.
classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss="binary_crossentropy",
    metrics=["binary_accuracy"]
)

classifier.summary()
print("✅ DebertaV3Classifier loaded and compiled successfully.")


I0000 00:00:1757389229.344611      19 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Preprocessor: "deberta_v3_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Tokenizer (type)                                   ┃                                             Vocab # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ deberta_v3_tokenizer (DebertaV3Tokenizer)          │                                             128,001 │
└────────────────────────────────────────────────────┴─────────────────────────────────────────────────────┘

Model: "deberta_v3_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ deberta_v3_backbone           │ (None, None, 768)         │     183,831,552 │ padding_mask[0][0],        │
│ (DebertaV3Backbone)           │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ get_item (GetItem)            │ (None, 768)               │               0 │ deberta_v3_backbone[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pooled_dropout (Dropout)      │ (None, 768)               │               0 │ get_item[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pooled_dense (Dense)          │ (None, 768)               │         590,592 │ pooled_dropout[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ classifier_dropout (Dropout)  │ (None, 768)               │               0 │ pooled_dense[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ logits (Dense)                │ (None, 65)                │          49,985 │ classifier_dropout[0][0]   │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 184,472,129 (703.71 MB)

 Trainable params: 184,472,129 (703.71 MB)

 Non-trainable params: 0 (0.00 B)

✅ DebertaV3Classifier loaded and compiled successfully.


In [7]:
# Cell 4: Manually Create TensorFlow Datasets
BATCH_SIZE = 8

# Create the training dataset from our pandas/numpy objects
tf_train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
# Shuffle, batch, and prefetch for optimal performance
tf_train_dataset = tf_train_dataset.shuffle(buffer_size=len(X_train)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Create the validation dataset
tf_val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
tf_val_dataset = tf_val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("✅ Data converted to the tf.data.Dataset format.")

✅ Data converted to the tf.data.Dataset format.


In [8]:
print("⏳ Starting model training...")

# The .fit() command starts the training process.
# - epochs: How many times the model will see the entire training dataset.
# - batch_size: How many samples the model works on at once.
# Cell 5: Train the Model
history = classifier.fit(
    tf_train_dataset,
    validation_data=tf_val_dataset,
    epochs=3
)

print("✅ Model training complete.")

print("✅ Model training complete.")

⏳ Starting model training...
Epoch 1/3


I0000 00:00:1757389303.298620      89 service.cc:152] XLA service 0x7ff028003680 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1757389303.298654      89 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1757389310.648756      89 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1757389357.979644      89 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


3670/3670 ━━━━━━━━━━━━━━━━━━━━ 2342s 606ms/step - binary_accuracy: 0.9838 - loss: 0.0894 - val_binary_accuracy: 0.9846 - val_loss: 0.0894
Epoch 2/3
3670/3670 ━━━━━━━━━━━━━━━━━━━━ 2155s 587ms/step - binary_accuracy: 0.9841 - loss: 0.0872 - val_binary_accuracy: 0.9846 - val_loss: 0.0873
Epoch 3/3
3670/3670 ━━━━━━━━━━━━━━━━━━━━ 2156s 587ms/step - binary_accuracy: 0.9840 - loss: 0.0847 - val_binary_accuracy: 0.9846 - val_loss: 0.0846
✅ Model training complete.
✅ Model training complete.


In [9]:
# Cell to add at the END of your Keras Training Notebook
import joblib
import os  # <-- Import the 'os' library

# Define a directory in the output folder to save everything
save_directory = "/kaggle/working/final_keras_model"

# NEW: Create the directory if it doesn't exist
os.makedirs(save_directory, exist_ok=True)

# Save the trained Keras model in the recommended .keras format
classifier.save(f"{save_directory}/model.keras")

# Save the MultiLabelBinarizer object, which we need for decoding predictions
joblib.dump(mlb, f"{save_directory}/multilabelbinarizer.joblib")

print(f"✅ All Keras assets (model, binarizer) saved to: {save_directory}")

✅ All Keras assets (model, binarizer) saved to: /kaggle/working/final_keras_model
